In [ ]:
# ============================================================
# CELL 1 — SETUP + GPU CHECK
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU             :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)
else:
    DEVICE = torch.device("cpu")
    print("WARNING: GPU not available")

print("Device          :", DEVICE)

print("=" * 60)

In [ ]:
# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

print("=" * 60)
print("GOOGLE DRIVE MOUNTED")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 3 — FIND DATASET
# ============================================================

import os

print("=" * 60)
print("GOOGLE DRIVE CONTENTS")
print("=" * 60)

base_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, "").count(os.sep)

    # Only show first few directory levels
    if level <= 2:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")

        for file in files[:10]:
            print(f"{indent}  {file}")

print("=" * 60)

In [ ]:
# ============================================================
# CELL 4 — DATASET PATHS + IMPORTS
# ============================================================

import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

DATASET_ROOT = "/content/drive/MyDrive/FINALCNNDATA_FINAL"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "validation")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

print("=" * 60)
print("DATASET PATH CHECK")
print("=" * 60)

print("Dataset root :", DATASET_ROOT)
print("Train exists :", os.path.exists(TRAIN_DIR))
print("Val exists   :", os.path.exists(VAL_DIR))
print("Test exists  :", os.path.exists(TEST_DIR))

print("=" * 60)

In [ ]:
# ============================================================
# CELL 5 — INSPECT DATASET FILES
# ============================================================

for name, folder in [
    ("TRAIN", TRAIN_DIR),
    ("VALIDATION", VAL_DIR),
    ("TEST", TEST_DIR)
]:

    files = []

    for ext in ["*.npy", "*.tif", "*.tiff"]:
        files.extend(glob.glob(os.path.join(folder, "**", ext), recursive=True))

    print(f"\n{name}")
    print("-" * 40)
    print("Files found:", len(files))

    for f in files[:5]:
        print(" ", os.path.basename(f))

print("\n" + "=" * 60)


TRAIN
----------------------------------------
Files found: 2549
  final_11_channel_stack_verified_fixed_039_042.tif
  final_11_channel_stack_verified_fixed_039_048.tif
  final_11_channel_stack_verified_fixed_039_052.tif
  final_11_channel_stack_verified_fixed_039_054.tif
  final_11_channel_stack_verified_fixed_039_050.tif

VALIDATION
----------------------------------------
Files found: 523
  final_11_channel_stack_verified_fixed_017_016.tif
  final_11_channel_stack_verified_fixed_018_013.tif
  final_11_channel_stack_verified_fixed_019_012.tif
  final_11_channel_stack_verified_fixed_019_013.tif
  final_11_channel_stack_verified_fixed_020_012.tif

TEST
----------------------------------------
Files found: 545
  final_11_channel_stack_verified_fixed_057_019.tif
  final_11_channel_stack_verified_fixed_057_017.tif
  final_11_channel_stack_verified_fixed_057_018.tif
  final_11_channel_stack_verified_fixed_059_020.tif
  final_11_channel_stack_verified_fixed_057_020.tif



In [ ]:
# ============================================================
# CELL 6 — INSPECT ONE PATCH
# ============================================================

train_files = []

for ext in ["*.npy", "*.tif", "*.tiff"]:
    train_files.extend(
        glob.glob(
            os.path.join(TRAIN_DIR, "**", ext),
            recursive=True
        )
    )

if len(train_files) == 0:
    raise RuntimeError("No training patches found!")

sample_file = train_files[0]

print("=" * 60)
print("SAMPLE PATCH CHECK")
print("=" * 60)

print("File:", sample_file)

if sample_file.lower().endswith(".npy"):
    sample = np.load(sample_file)

else:
    import rasterio

    with rasterio.open(sample_file) as src:
        sample = src.read()

sample = np.asarray(sample)

print("Shape :", sample.shape)
print("Dtype :", sample.dtype)
print("Min   :", np.nanmin(sample))
print("Max   :", np.nanmax(sample))

print("=" * 60)

# Expected:
# (11, 250, 250)

SAMPLE PATCH CHECK
File: /content/drive/MyDrive/FINALCNNDATA_FINAL/train/no_site/final_11_channel_stack_verified_fixed_039_042.tif
Shape : (11, 250, 250)
Dtype : float32
Min   : -1.4677685
Max   : 48.623917


In [ ]:
# ============================================================
# CELL 7 — DATASET CLASS
# ============================================================

import os
import glob
import numpy as np
import torch
import rasterio
from torch.utils.data import Dataset

class ArchaeologyDataset(Dataset):

    def __init__(self, root_dir):

        self.files = sorted(
            glob.glob(
                os.path.join(root_dir, "**", "*.tif"),
                recursive=True
            )
        )

        if len(self.files) == 0:
            raise RuntimeError(f"No TIFF files found in {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        filepath = self.files[idx]

        with rasterio.open(filepath) as src:
            image = src.read().astype(np.float32)

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------
        if image.shape != (11, 250, 250):
            raise ValueError(
                f"Unexpected shape {image.shape} in {filepath}"
            )

        # ----------------------------------------------------
        # Determine label from folder name
        # ----------------------------------------------------
        parent_folder = os.path.basename(
            os.path.dirname(filepath)
        ).lower()

        if parent_folder in ["site", "positive", "pos"]:
            label = 1.0

        elif parent_folder in ["no_site", "negative", "neg"]:
            label = 0.0

        else:
            raise ValueError(
                f"Unknown label folder: {parent_folder}"
            )

        image = torch.from_numpy(image)
        label = torch.tensor(label, dtype=torch.float32)

        return image, label


print("=" * 60)
print("DATASET CLASS READY")
print("=" * 60)

DATASET CLASS READY


In [ ]:
# ============================================================
# CELL 8 — CREATE DATASETS
# ============================================================

train_dataset = ArchaeologyDataset(TRAIN_DIR)
val_dataset   = ArchaeologyDataset(VAL_DIR)
test_dataset  = ArchaeologyDataset(TEST_DIR)

print("=" * 60)
print("DATASET SIZES")
print("=" * 60)

print("Train       :", len(train_dataset))
print("Validation  :", len(val_dataset))
print("Test        :", len(test_dataset))

print("=" * 60)

DATASET SIZES
Train       : 2549
Validation  : 523
Test        : 545


In [ ]:
# ============================================================
# CELL 9 — DATALOADERS + BATCH CHECK
# ============================================================

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Check one batch
# ------------------------------------------------------------

batch_inputs, batch_labels = next(iter(train_loader))

print("=" * 60)
print("DATALOADER CHECK")
print("=" * 60)

print("Input shape :", batch_inputs.shape)
print("Label shape :", batch_labels.shape)

print("Input dtype :", batch_inputs.dtype)
print("Label dtype :", batch_labels.dtype)

print("Input min   :", batch_inputs.min().item())
print("Input max   :", batch_inputs.max().item())

print("Labels      :", batch_labels[:10])

print("=" * 60)

print("Expected input shape:")
print("[16, 11, 250, 250]")

DATALOADER CHECK
Input shape : torch.Size([16, 11, 250, 250])
Label shape : torch.Size([16])
Input dtype : torch.float32
Label dtype : torch.float32
Input min   : -9999.0
Input max   : 74.11305236816406
Labels      : tensor([1., 0., 1., 1., 1., 1., 0., 0., 1., 0.])
Expected input shape:
[16, 11, 250, 250]


In [ ]:
# ============================================================
# CELL 10 — COMPUTE TRAINING NORMALIZATION STATISTICS
# ============================================================

import random
import numpy as np
import rasterio

print("=" * 60)
print("COMPUTING CHANNEL NORMALIZATION STATISTICS")
print("=" * 60)

# Use a representative subset so this does not take forever
random.seed(42)

num_stat_samples = min(200, len(train_dataset))

stat_indices = random.sample(
    range(len(train_dataset)),
    num_stat_samples
)

channel_sum = np.zeros(11, dtype=np.float64)
channel_sq_sum = np.zeros(11, dtype=np.float64)
channel_count = np.zeros(11, dtype=np.int64)

for i, idx in enumerate(stat_indices):

    filepath = train_dataset.files[idx]

    with rasterio.open(filepath) as src:
        image = src.read().astype(np.float32)

    for c in range(11):

        channel = image[c]

        # Ignore NoData
        valid = np.isfinite(channel) & (channel > -9000)

        values = channel[valid].astype(np.float64)

        if len(values) == 0:
            continue

        channel_sum[c] += values.sum()
        channel_sq_sum[c] += np.square(values).sum()
        channel_count[c] += len(values)

channel_mean = channel_sum / channel_count

channel_var = (
    channel_sq_sum / channel_count
    - channel_mean ** 2
)

channel_std = np.sqrt(
    np.maximum(channel_var, 1e-8)
)

print("Samples used :", num_stat_samples)
print()

for c in range(11):
    print(
        f"Channel {c+1:02d} | "
        f"Mean: {channel_mean[c]:10.5f} | "
        f"Std: {channel_std[c]:10.5f} | "
        f"Valid: {channel_count[c]}"
    )

print("=" * 60)

COMPUTING CHANNEL NORMALIZATION STATISTICS
Samples used : 200

Channel 01 | Mean:    0.03493 | Std:    0.00822 | Valid: 12487172
Channel 02 | Mean:    0.04262 | Std:    0.01646 | Valid: 12487172
Channel 03 | Mean:    0.03490 | Std:    0.01402 | Valid: 12487172
Channel 04 | Mean:    0.25016 | Std:    0.09953 | Valid: 12487172
Channel 05 | Mean:    0.01882 | Std:    1.09393 | Valid: 12487172
Channel 06 | Mean:    7.59750 | Std:    6.45154 | Valid: 12487172
Channel 07 | Mean:    0.92841 | Std:    0.05365 | Valid: 12487172
Channel 08 | Mean:    0.63347 | Std:    0.09198 | Valid: 12474202
Channel 09 | Mean:    0.62814 | Std:    0.09409 | Valid: 12474202
Channel 10 | Mean:    0.63306 | Std:    0.09137 | Valid: 12474202
Channel 11 | Mean:    0.63855 | Std:    0.09426 | Valid: 12474202


In [ ]:
# ============================================================
# CELL 11 — NORMALIZED DATASET
# ============================================================

class ArchaeologyDataset(Dataset):

    def __init__(self, root_dir, mean, std):

        self.files = sorted(
            glob.glob(
                os.path.join(root_dir, "**", "*.tif"),
                recursive=True
            )
        )

        if len(self.files) == 0:
            raise RuntimeError(
                f"No TIFF files found in {root_dir}"
            )

        self.mean = torch.tensor(
            mean,
            dtype=torch.float32
        ).view(11, 1, 1)

        self.std = torch.tensor(
            std,
            dtype=torch.float32
        ).view(11, 1, 1)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        filepath = self.files[idx]

        with rasterio.open(filepath) as src:
            image = src.read().astype(np.float32)

        if image.shape != (11, 250, 250):
            raise ValueError(
                f"Unexpected shape {image.shape}"
            )

        # ----------------------------------------------------
        # Label
        # ----------------------------------------------------

        parent_folder = os.path.basename(
            os.path.dirname(filepath)
        ).lower()

        if parent_folder in ["site", "positive", "pos"]:
            label = 1.0

        elif parent_folder in ["no_site", "negative", "neg"]:
            label = 0.0

        else:
            raise ValueError(
                f"Unknown label folder: {parent_folder}"
            )

        # ----------------------------------------------------
        # Convert to tensor
        # ----------------------------------------------------

        image = torch.from_numpy(image)

        # ----------------------------------------------------
        # NoData handling
        # ----------------------------------------------------

        nodata_mask = (
            ~torch.isfinite(image)
            | (image <= -9000)
        )

        # ----------------------------------------------------
        # Replace NoData temporarily with channel mean
        # ----------------------------------------------------

        for c in range(11):

            image[c][nodata_mask[c]] = self.mean[c, 0, 0]

        # ----------------------------------------------------
        # Channel-wise standardization
        # ----------------------------------------------------

        image = (
            image - self.mean
        ) / self.std

        # ----------------------------------------------------
        # Set originally missing pixels to 0 after
        # normalization.
        #
        # 0 now means "channel mean / neutral value"
        # ----------------------------------------------------

        image[nodata_mask] = 0.0

        label = torch.tensor(
            label,
            dtype=torch.float32
        )

        return image, label


print("=" * 60)
print("NORMALIZED DATASET CLASS READY")
print("=" * 60)

NORMALIZED DATASET CLASS READY


In [ ]:
# ============================================================
# CELL 12 — RECREATE DATASETS + DATALOADERS
# ============================================================

train_dataset = ArchaeologyDataset(
    TRAIN_DIR,
    channel_mean,
    channel_std
)

val_dataset = ArchaeologyDataset(
    VAL_DIR,
    channel_mean,
    channel_std
)

test_dataset = ArchaeologyDataset(
    TEST_DIR,
    channel_mean,
    channel_std
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

batch_inputs, batch_labels = next(
    iter(train_loader)
)

print("=" * 60)
print("NORMALIZED DATALOADER CHECK")
print("=" * 60)

print("Input shape :", batch_inputs.shape)
print("Label shape :", batch_labels.shape)

print("Input dtype :", batch_inputs.dtype)
print("Label dtype :", batch_labels.dtype)

print("Input min   :", batch_inputs.min().item())
print("Input max   :", batch_inputs.max().item())

print("Input mean  :", batch_inputs.mean().item())
print("Input std   :", batch_inputs.std().item())

print("Labels      :", batch_labels[:10])

print("=" * 60)

print("Expected:")
print("[16, 11, 250, 250]")
print("No -9999 values should remain.")

NORMALIZED DATALOADER CHECK
Input shape : torch.Size([16, 11, 250, 250])
Label shape : torch.Size([16])
Input dtype : torch.float32
Label dtype : torch.float32
Input min   : -12.02015495300293
Input max   : 26.323823928833008
Input mean  : 0.008299804292619228
Input std   : 1.0251425504684448
Labels      : tensor([0., 1., 0., 1., 0., 0., 0., 0., 0., 0.])
Expected:
[16, 11, 250, 250]
No -9999 values should remain.


In [ ]:
# ============================================================
# CELL 13 — CNN FEATURE EXTRACTOR
# ============================================================

import torch
import torch.nn as nn


class TerrainCNN(nn.Module):

    def __init__(self, in_channels=11):

        super().__init__()

        # ----------------------------------------------------
        # BLOCK 1
        # 11 -> 32
        # ----------------------------------------------------

        self.block1 = nn.Sequential(

            nn.Conv2d(
                in_channels,
                32,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm2d(32),

            nn.GELU(),

            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(32),

            nn.GELU()
        )

        # ----------------------------------------------------
        # BLOCK 2
        # 32 -> 64
        # ----------------------------------------------------

        self.block2 = nn.Sequential(

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(64),

            nn.GELU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(64),

            nn.GELU()
        )

        # ----------------------------------------------------
        # BLOCK 3
        # 64 -> 128
        # ----------------------------------------------------

        self.block3 = nn.Sequential(

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(128),

            nn.GELU(),

            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(128),

            nn.GELU()
        )

        # ----------------------------------------------------
        # DOWNSAMPLING
        # ----------------------------------------------------

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # ----------------------------------------------------
        # FINAL PROJECTION
        # 128 -> 64
        #
        # MobileViT receives 64 channels
        # ----------------------------------------------------

        self.projection = nn.Sequential(

            nn.Conv2d(
                128,
                64,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(64),

            nn.GELU()
        )

    def forward(self, x):

        # 250 x 250
        x = self.block1(x)

        # 125 x 125
        x = self.pool(x)

        x = self.block2(x)

        # 62 x 62
        x = self.pool(x)

        x = self.block3(x)

        # 31 x 31
        x = self.pool(x)

        # 64 channels for MobileViT
        x = self.projection(x)

        return x


print("=" * 60)
print("CNN CREATED")
print("=" * 60)

cnn = TerrainCNN(
    in_channels=11
).to(DEVICE)

total_params = sum(
    p.numel()
    for p in cnn.parameters()
)

trainable_params = sum(
    p.numel()
    for p in cnn.parameters()
    if p.requires_grad
)

print("CNN parameters :", total_params)
print("Trainable      :", trainable_params)

print("=" * 60)

CNN CREATED
CNN parameters : 303712
Trainable      : 303712


In [ ]:
# ============================================================
# CELL 14 — CNN FORWARD PASS CHECK
# ============================================================

cnn.eval()

with torch.no_grad():

    test_batch = batch_inputs.to(DEVICE)

    cnn_features = cnn(test_batch)

print("=" * 60)
print("CNN FORWARD PASS CHECK")
print("=" * 60)

print("Input shape      :", test_batch.shape)
print("CNN output shape :", cnn_features.shape)

print()
print("Expected:")
print("[16, 64, 31, 31]")

print()
print("Output dtype :", cnn_features.dtype)
print("Output min   :", cnn_features.min().item())
print("Output max   :", cnn_features.max().item())

print("=" * 60)

CNN FORWARD PASS CHECK
Input shape      : torch.Size([16, 11, 250, 250])
CNN output shape : torch.Size([16, 64, 31, 31])

Expected:
[16, 64, 31, 31]

Output dtype : torch.float32
Output min   : -0.0031195550691336393
Output max   : 0.003846572246402502


In [ ]:
# ============================================================
# CELL 15 — MOBILEVIT BLOCK
# ============================================================

class TransformerEncoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        num_heads=4,
        mlp_ratio=2.0,
        dropout=0.1
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(
            dim * mlp_ratio
        )

        self.mlp = nn.Sequential(

            nn.Linear(dim, hidden_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, dim),

            nn.Dropout(dropout)
        )

    def forward(self, x):

        # Self-attention
        residual = x

        x = self.norm1(x)

        x, _ = self.attention(
            x, x, x
        )

        x = x + residual

        # Feed-forward
        residual = x

        x = self.norm2(x)

        x = self.mlp(x)

        x = x + residual

        return x


class MobileViTBlock(nn.Module):

    def __init__(
        self,
        in_channels=64,
        transformer_dim=96,
        patch_size=4,
        num_heads=4,
        num_layers=2,
        mlp_ratio=2.0,
        dropout=0.1
    ):

        super().__init__()

        self.local_rep = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(in_channels),

            nn.GELU(),

            nn.Conv2d(
                in_channels,
                transformer_dim,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(transformer_dim),

            nn.GELU()
        )

        self.transformer = nn.ModuleList([

            TransformerEncoderBlock(
                dim=transformer_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                dropout=dropout
            )

            for _ in range(num_layers)

        ])

        self.project = nn.Sequential(

            nn.Conv2d(
                transformer_dim,
                in_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(in_channels),

            nn.GELU()
        )

    def forward(self, x):

        residual = x

        x = self.local_rep(x)

        B, C, H, W = x.shape

        # ----------------------------------------------------
        # Convert feature map into tokens
        # ----------------------------------------------------

        x = x.flatten(2)

        x = x.transpose(1, 2)

        # [B, H*W, C]

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        for block in self.transformer:

            x = block(x)

        # ----------------------------------------------------
        # Tokens -> feature map
        # ----------------------------------------------------

        x = x.transpose(1, 2)

        x = x.reshape(
            B,
            C,
            H,
            W
        )

        x = self.project(x)

        # Residual fusion
        x = x + residual

        return x


print("=" * 60)
print("MOBILEVIT BLOCK CREATED")
print("=" * 60)

MOBILEVIT BLOCK CREATED


In [ ]:
# ============================================================
# CELL 16 — CREATE MOBILEVIT + FORWARD PASS
# ============================================================

mobilevit = MobileViTBlock(
    in_channels=64,
    transformer_dim=96,
    patch_size=4,
    num_heads=4,
    num_layers=2,
    mlp_ratio=2.0,
    dropout=0.10
).to(DEVICE)

mobilevit.eval()

with torch.no_grad():
    mobilevit_features = mobilevit(cnn_features)

print("=" * 60)
print("CNN -> MOBILEVIT FORWARD PASS")
print("=" * 60)

print("CNN input shape      :", cnn_features.shape)
print("MobileViT output     :", mobilevit_features.shape)

print()
print("Expected:")
print("[16, 64, 31, 31]")

print()
print("dtype :", mobilevit_features.dtype)
print("min   :", mobilevit_features.min().item())
print("max   :", mobilevit_features.max().item())

print("=" * 60)

CNN -> MOBILEVIT FORWARD PASS
CNN input shape      : torch.Size([16, 64, 31, 31])
MobileViT output     : torch.Size([16, 64, 31, 31])

Expected:
[16, 64, 31, 31]

dtype : torch.float32
min   : -0.17151783406734467
max   : 0.6400951147079468


In [ ]:
# ============================================================
# CELL 17 — CLASSIFIER
# ============================================================

class ArchaeologyClassifier(nn.Module):

    def __init__(self, in_channels=64):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(

            nn.Linear(in_channels, 32),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x


classifier = ArchaeologyClassifier(
    in_channels=64
).to(DEVICE)


# Quick check
classifier.eval()

with torch.no_grad():

    logits = classifier(
        mobilevit_features
    )

print("=" * 60)
print("CLASSIFIER CHECK")
print("=" * 60)

print("MobileViT features :", mobilevit_features.shape)
print("Logits              :", logits.shape)

print("Expected             : [16, 1]")

print("Logit min            :", logits.min().item())
print("Logit max            :", logits.max().item())

print("=" * 60)

CLASSIFIER CHECK
MobileViT features : torch.Size([16, 64, 31, 31])
Logits              : torch.Size([16, 1])
Expected             : [16, 1]
Logit min            : -0.026192206889390945
Logit max            : 0.01176464930176735


In [ ]:
# ============================================================
# CELL 17.5 — SET DEVICE
# ============================================================
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)
if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    print("WARNING: CUDA is not available.")

Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 18 — CNN FEATURE MAP → 128-CHANNEL MOBILEVIT INPUT
# ============================================================
class CNNToMobileViT128(nn.Module):
    def __init__(self):
        super().__init__()
        # ----------------------------------------------------
        # Learn a richer representation from the CNN feature map
        #
        # Input:
        #   [B, 64, 31, 31]
        #
        # Output:
        #   [B, 128, 31, 31]
        # ----------------------------------------------------
        self.feature_adapter = nn.Sequential(
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(
                in_channels=128,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(128),
            nn.GELU()
        )
        # ----------------------------------------------------
        # Upsampling
        #
        # 31 × 31 → 256 × 256
        # ----------------------------------------------------
    def forward(self, x):
        # x:
        # [B, 64, 31, 31]
        x = self.feature_adapter(x)

        # [B, 128, 31, 31]
        x = F.interpolate(
            x,
            size=(256, 256),
            mode="bilinear",
            align_corners=False
        )
        # [B, 128, 256, 256]
        return x
adapter = CNNToMobileViT128().to(device)
print("128-channel CNN → MobileViT adapter created.")

128-channel CNN → MobileViT adapter created.


In [ ]:
# ============================================================
# CELL 19 — INSPECT HUGGING FACE MOBILEVIT STRUCTURE
# ============================================================
print("=" * 70)
print("MOBILEVIT MODEL STRUCTURE")
print("=" * 70)

print("\nModel type:")
print(type(mobilevit))

print("\nTop-level children:")
for name, module in mobilevit.named_children():
    print(f"{name} -> {type(module).__name__}")

print("\nFull model structure:")
print(mobilevit)

MOBILEVIT MODEL STRUCTURE

Model type:
<class '__main__.MobileViTBlock'>

Top-level children:
local_rep -> Sequential
transformer -> ModuleList
project -> Sequential

Full model structure:
MobileViTBlock(
  (local_rep): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Conv2d(64, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (4): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
  )
  (transformer): ModuleList(
    (0-1): 2 x TransformerEncoderBlock(
      (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
      )
      (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
 

In [ ]:
# ============================================================
# CELL 19 — CNN FEATURE → 128-CHANNEL HIDDEN CNN → 64 CHANNEL
#              → PRETRAINED MOBILEVIT
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class CNNToMobileViT(nn.Module):

    def __init__(self):
        super().__init__()

        # ----------------------------------------------------
        # Hidden CNN
        # Input from frozen CNN:
        # [B, 64, 31, 31]
        #
        # We temporarily expand the representation:
        # 64 → 128
        # ----------------------------------------------------

        self.hidden_cnn = nn.Sequential(

            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(128),

            nn.GELU(),

            nn.Conv2d(
                in_channels=128,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(128),

            nn.GELU()
        )

        # ----------------------------------------------------
        # Project 128 → 64
        #
        # This is necessary because the pretrained MobileViT
        # block expects exactly 64 input channels.
        # ----------------------------------------------------

        self.mobilevit_adapter = nn.Sequential(

            nn.Conv2d(
                in_channels=128,
                out_channels=64,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False
            ),

            nn.BatchNorm2d(64),

            nn.GELU()
        )


    def forward(self, x):

        # x:
        # [B, 64, 31, 31]

        x = self.hidden_cnn(x)

        # [B, 128, 31, 31]

        x = self.mobilevit_adapter(x)

        # [B, 64, 31, 31]

        return x


# ------------------------------------------------------------
# Create adapter
# ------------------------------------------------------------

cnn_to_mobilevit = CNNToMobileViT().to(device)

print("CNN → MobileViT adapter created.")

print()
print("Adapter structure:")
print(cnn_to_mobilevit)

CNN → MobileViT adapter created.

Adapter structure:
CNNToMobileViT(
  (hidden_cnn): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
  )
  (mobilevit_adapter): Sequential(
    (0): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
)


In [ ]:
# ============================================================
# CELL 20 — CHECK CNN OUTPUT
# ============================================================

cnn.eval()
cnn_to_mobilevit.eval()

# ------------------------------------------------------------
# Get a fresh batch
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Input images:")
print(images.shape)

print("Labels:")
print(labels.shape)


# ------------------------------------------------------------
# Run CNN WITHOUT return_features
# ------------------------------------------------------------

with torch.no_grad():

    cnn_output = cnn(images)


# ------------------------------------------------------------
# Inspect what CNN actually returns
# ------------------------------------------------------------

print()
print("CNN output type:")
print(type(cnn_output))

if isinstance(cnn_output, tuple):

    print("CNN returned", len(cnn_output), "outputs")

    for i, item in enumerate(cnn_output):
        print(
            f"Output {i}:",
            item.shape if hasattr(item, "shape") else type(item)
        )

else:

    print(
        "CNN output shape:",
        cnn_output.shape
    )

Input images:
torch.Size([16, 11, 250, 250])
Labels:
torch.Size([16])

CNN output type:
<class 'torch.Tensor'>
CNN output shape: torch.Size([16, 64, 31, 31])


In [ ]:
# ============================================================
# CELL 20 — CNN → HIDDEN CNN → MOBILEVIT ADAPTER
# ============================================================

cnn.eval()
cnn_to_mobilevit.eval()

# ------------------------------------------------------------
# Get a fresh batch
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Input images:")
print(images.shape)

print("Labels:")
print(labels.shape)


# ------------------------------------------------------------
# STEP 1 — Frozen trained CNN
# ------------------------------------------------------------

with torch.no_grad():

    cnn_features = cnn(images)

print()
print("CNN feature map:")
print(cnn_features.shape)


# ------------------------------------------------------------
# STEP 2 — Hidden CNN
#
# 64 channels → 128 channels
#
# Spatial resolution remains:
# 31 × 31
# ------------------------------------------------------------

with torch.no_grad():

    hidden_features = cnn_to_mobilevit.hidden_cnn(
        cnn_features
    )

print()
print("After hidden CNN:")
print(hidden_features.shape)


# ------------------------------------------------------------
# STEP 3 — 128 → 64 channel adapter
# ------------------------------------------------------------

with torch.no_grad():

    mobilevit_features = (
        cnn_to_mobilevit.mobilevit_adapter(
            hidden_features
        )
    )

print()
print("Final MobileViT feature map:")
print(mobilevit_features.shape)

Input images:
torch.Size([16, 11, 250, 250])
Labels:
torch.Size([16])

CNN feature map:
torch.Size([16, 64, 31, 31])

After hidden CNN:
torch.Size([16, 128, 31, 31])

Final MobileViT feature map:
torch.Size([16, 64, 31, 31])


In [ ]:
# ============================================================
# FINAL 64 → 3 CHANNEL ADAPTER
# ============================================================

import torch.nn as nn

final_channel_adapter = nn.Sequential(

    # 64 → 32
    nn.Conv2d(
        64, 32,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    ),
    nn.BatchNorm2d(32),
    nn.GELU(),

    # 32 → 16
    nn.Conv2d(
        32, 16,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    ),
    nn.BatchNorm2d(16),
    nn.GELU(),

    # 16 → 3
    nn.Conv2d(
        16, 3,
        kernel_size=1,
        stride=1,
        padding=0,
        bias=True
    )
).to(device)

print("Final 64 → 3 adapter created.")

Final 64 → 3 adapter created.


In [ ]:
# ============================================================
# TEST CNN → HIDDEN CNN → 64 → 3
# ============================================================

cnn.eval()
cnn_to_mobilevit.eval()
final_channel_adapter.eval()

images = images.to(device)

with torch.no_grad():

    # --------------------------------------------------------
    # Step 1 — Frozen CNN
    # --------------------------------------------------------

    cnn_features = cnn(images)

    print("CNN features:")
    print(cnn_features.shape)


    # --------------------------------------------------------
    # Step 2 — Hidden CNN
    # --------------------------------------------------------

    hidden_features = cnn_to_mobilevit.hidden_cnn(
        cnn_features
    )

    print()
    print("After hidden CNN:")
    print(hidden_features.shape)


    # --------------------------------------------------------
    # Step 3 — 128 → 64 adapter
    # --------------------------------------------------------

    features_64 = cnn_to_mobilevit.mobilevit_adapter(
        hidden_features
    )

    print()
    print("After 128 → 64 adapter:")
    print(features_64.shape)

    # --------------------------------------------------------
    # Step 4 — 64 → 3 final adapter
    # --------------------------------------------------------

    features_3 = final_channel_adapter(
        features_64
    )

    print()
    print("After final 64 → 3 adapter:")
    print(features_3.shape)

    # --------------------------------------------------------
    # Step 5 — 31 × 31 → 256 × 256
    # --------------------------------------------------------

    mobilevit_input = F.interpolate(
        features_3,
        size=(256, 256),
        mode="bilinear",
        align_corners=False
    )

    print()
    print("Final MobileViT input:")
    print(mobilevit_input.shape)

CNN features:
torch.Size([16, 64, 31, 31])

After hidden CNN:
torch.Size([16, 128, 31, 31])

After 128 → 64 adapter:
torch.Size([16, 64, 31, 31])

After final 64 → 3 adapter:
torch.Size([16, 3, 31, 31])

Final MobileViT input:
torch.Size([16, 3, 256, 256])


In [ ]:
# ============================================================
# CELL 26 — LOAD TRAINED CNN AS FROZEN FEATURE EXTRACTOR
# ============================================================

import torch
import torch.nn as nn


# ------------------------------------------------------------
# CNN ARCHITECTURE
# EXACTLY MATCHES best_cnn_model.pth
# ------------------------------------------------------------

class CNNBaseline(nn.Module):

    def __init__(self, in_channels=11):

        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                16,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Linear(64, 1)


    def forward(self, x):

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        # ----------------------------------------------------
        # IMPORTANT
        #
        # Return the 64-channel spatial feature map.
        #
        # Shape:
        # [B, 64, 31, 31]
        # ----------------------------------------------------

        return x


# ------------------------------------------------------------
# LOAD TRAINED CNN
# ------------------------------------------------------------

CNN_CHECKPOINT = (
    "/content/drive/MyDrive/"
    "best_cnn_model.pth"
)

cnn = CNNBaseline(
    in_channels=11
).to(device)


checkpoint = torch.load(
    CNN_CHECKPOINT,
    map_location=device
)

cnn.load_state_dict(checkpoint)


# ------------------------------------------------------------
# FREEZE CNN
# ------------------------------------------------------------

cnn.eval()

for param in cnn.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# VERIFY CNN FEATURE MAP
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    cnn_features = cnn(images)


print("=" * 70)
print("TRAINED CNN LOADED SUCCESSFULLY")
print("=" * 70)

print("Checkpoint:")
print(CNN_CHECKPOINT)

print()
print("Input:")
print(images.shape)

print()
print("CNN feature map:")
print(cnn_features.shape)

print()
print("Expected:")
print("[B, 64, 31, 31]")

print()
print(
    "Frozen CNN parameters:",
    sum(
        p.numel()
        for p in cnn.parameters()
        if not p.requires_grad
    )
)

print("=" * 70)

TRAINED CNN LOADED SUCCESSFULLY
Checkpoint:
/content/drive/MyDrive/best_cnn_model.pth

Input:
torch.Size([16, 11, 250, 250])

CNN feature map:
torch.Size([16, 64, 31, 31])

Expected:
[B, 64, 31, 31]

Frozen CNN parameters: 25025


In [ ]:
# ============================================================
# CELL 26.5 — LOAD PRETRAINED HUGGING FACE MOBILEVIT
# ============================================================

from transformers import (
    MobileViTForImageClassification
)


# ------------------------------------------------------------
# Load pretrained MobileViT
# ------------------------------------------------------------

HF_MODEL_NAME = "apple/mobilevit-small"


hf_mobilevit = (
    MobileViTForImageClassification
    .from_pretrained(HF_MODEL_NAME)
)


# ------------------------------------------------------------
# Move to device
# ------------------------------------------------------------

hf_mobilevit = hf_mobilevit.to(device)


# ------------------------------------------------------------
# Replace classifier
#
# Pretrained classifier:
# 640 → 1000
#
# Our task:
# binary archaeological classification
#
# Therefore:
# 640 → 1
# ------------------------------------------------------------

hf_mobilevit.classifier = nn.Linear(
    hf_mobilevit.classifier.in_features,
    1
).to(device)


# ------------------------------------------------------------
# Print verification
# ------------------------------------------------------------

print("=" * 70)
print("PRETRAINED HUGGING FACE MOBILEVIT LOADED")
print("=" * 70)

print()
print("Model type:")
print(type(hf_mobilevit))

print()
print("Device:")
print(next(hf_mobilevit.parameters()).device)

print()
print("Input channels expected:")
print(hf_mobilevit.config.num_channels)

print()
print("Classifier:")
print(hf_mobilevit.classifier)

print()
print("Classifier output features:")
print(hf_mobilevit.classifier.out_features)

print("=" * 70)

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 22.5MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/347 [00:00<?, ?it/s]

PRETRAINED HUGGING FACE MOBILEVIT LOADED

Model type:
<class 'transformers.models.mobilevit.modeling_mobilevit.MobileViTForImageClassification'>

Device:
cuda:0

Input channels expected:
3

Classifier:
Linear(in_features=640, out_features=1, bias=True)

Classifier output features:
1


In [ ]:
# ============================================================
# CELL 27 — SET TRAINABLE / FROZEN COMPONENTS
# ============================================================

print("=" * 70)
print("SETTING TRAINABLE COMPONENTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. CNN = FROZEN
# ------------------------------------------------------------

cnn.eval()

for param in cnn.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# 2. Hidden CNN + 128→64 adapter = TRAINABLE
# ------------------------------------------------------------

cnn_to_mobilevit.hidden_cnn.train()

for param in cnn_to_mobilevit.hidden_cnn.parameters():
    param.requires_grad = True


cnn_to_mobilevit.mobilevit_adapter.train()

for param in cnn_to_mobilevit.mobilevit_adapter.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 3. Final 64→3 adapter = TRAINABLE
# ------------------------------------------------------------

final_channel_adapter.train()

for param in final_channel_adapter.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 4. Hugging Face MobileViT
# ------------------------------------------------------------

# Freeze entire pretrained MobileViT
for param in hf_mobilevit.parameters():
    param.requires_grad = False


# Only classifier is trainable
for param in hf_mobilevit.classifier.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# IMPORTANT:
# Keep classifier in train mode.
# Keep pretrained backbone frozen.
# ------------------------------------------------------------

hf_mobilevit.classifier.train()


# ------------------------------------------------------------
# PRINT PARAMETER SUMMARY
# ------------------------------------------------------------

cnn_params = sum(
    p.numel()
    for p in cnn.parameters()
)

hidden_cnn_params = sum(
    p.numel()
    for p in cnn_to_mobilevit.hidden_cnn.parameters()
)

adapter_params = sum(
    p.numel()
    for p in cnn_to_mobilevit.mobilevit_adapter.parameters()
)

final_adapter_params = sum(
    p.numel()
    for p in final_channel_adapter.parameters()
)

hf_total_params = sum(
    p.numel()
    for p in hf_mobilevit.parameters()
)

hf_trainable_params = sum(
    p.numel()
    for p in hf_mobilevit.parameters()
    if p.requires_grad
)

total_trainable = (
    hidden_cnn_params
    + adapter_params
    + final_adapter_params
    + hf_trainable_params
)

print()
print("=" * 70)
print("PARAMETER SUMMARY")
print("=" * 70)

print(f"CNN parameters              : {cnn_params:,}")
print(f"Hidden CNN parameters       : {hidden_cnn_params:,}")
print(f"128 → 64 adapter parameters : {adapter_params:,}")
print(f"64 → 3 adapter parameters   : {final_adapter_params:,}")
print(f"HF MobileViT total          : {hf_total_params:,}")
print(f"HF MobileViT trainable      : {hf_trainable_params:,}")

print()
print(f"TOTAL TRAINABLE PARAMETERS  : {total_trainable:,}")

SETTING TRAINABLE COMPONENTS

PARAMETER SUMMARY
CNN parameters              : 25,025
Hidden CNN parameters       : 221,696
128 → 64 adapter parameters : 8,320
64 → 3 adapter parameters   : 23,187
HF MobileViT total          : 4,938,273
HF MobileViT trainable      : 641

TOTAL TRAINABLE PARAMETERS  : 253,844


In [ ]:
# ============================================================
# CELL 28 — OPTIMIZER
# ============================================================

import torch

# ------------------------------------------------------------
# Collect ALL trainable parameters
# ------------------------------------------------------------

trainable_parameters = (
    list(cnn_to_mobilevit.hidden_cnn.parameters())
    +
    list(cnn_to_mobilevit.mobilevit_adapter.parameters())
    +
    list(final_channel_adapter.parameters())
    +
    list(hf_mobilevit.classifier.parameters())
)

# ------------------------------------------------------------
# AdamW
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-4,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

criterion = torch.nn.BCEWithLogitsLoss()

# ------------------------------------------------------------
# Verify optimizer
# ------------------------------------------------------------

optimizer_params = sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
)

print("=" * 70)
print("MOBILEVIT TRAINING SETUP")
print("=" * 70)

print()
print("Loss function:")
print(criterion)

print()
print("Optimizer:")
print(optimizer)

print()
print("Learning rate:", 1e-4)
print("Weight decay:", 1e-4)

print()
print("Trainable parameters in optimizer:")
print(f"{optimizer_params:,}")

MOBILEVIT TRAINING SETUP

Loss function:
BCEWithLogitsLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)

Learning rate: 0.0001
Weight decay: 0.0001

Trainable parameters in optimizer:
253,844


In [ ]:
# ============================================================
# CELL 29 — TRAIN CNN → HIDDEN CNN → 64 → 3 → MOBILEVIT
# ============================================================

import torch
import torch.nn.functional as F

NUM_EPOCHS = 15

best_val_loss = float("inf")

best_model_path = (
    "/content/drive/MyDrive/"
    "best_cnn_hidden_mobilevit.pth"
)

print("=" * 70)
print("STARTING CNN → HIDDEN CNN → ADAPTER → MOBILEVIT TRAINING")
print("=" * 70)


for epoch in range(NUM_EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    cnn.eval()

    cnn_to_mobilevit.hidden_cnn.train()
    cnn_to_mobilevit.mobilevit_adapter.train()
    final_channel_adapter.train()
    hf_mobilevit.classifier.train()

    train_loss = 0.0
    train_samples = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        # ----------------------------------------------------
        # STEP 1 — FROZEN CNN
        # ----------------------------------------------------

        with torch.no_grad():

            cnn_features = cnn(images)

        # ----------------------------------------------------
        # STEP 2 — HIDDEN CNN
        # 64 → 128
        # ----------------------------------------------------

        hidden_features = (
            cnn_to_mobilevit.hidden_cnn(
                cnn_features
            )
        )

        # ----------------------------------------------------
        # STEP 3 — 128 → 64
        # ----------------------------------------------------

        features_64 = (
            cnn_to_mobilevit.mobilevit_adapter(
                hidden_features
            )
        )

        # ----------------------------------------------------
        # STEP 4 — 64 → 3
        # ----------------------------------------------------

        features_3 = (
            final_channel_adapter(
                features_64
            )
        )

        # ----------------------------------------------------
        # STEP 5 — 31×31 → 256×256
        # ----------------------------------------------------

        mobilevit_input = F.interpolate(
            features_3,
            size=(256, 256),
            mode="bilinear",
            align_corners=False
        )

        # ----------------------------------------------------
        # STEP 6 — PRETRAINED MOBILEVIT
        # ----------------------------------------------------

        outputs = hf_mobilevit(
            pixel_values=mobilevit_input
        )

        logits = outputs.logits.squeeze(1)

        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss = criterion(
            logits,
            labels
        )

        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()

        optimizer.step()

        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        batch_size = images.size(0)

        train_loss += loss.item() * batch_size
        train_samples += batch_size

    train_loss /= train_samples


    # ========================================================
    # VALIDATION
    # ========================================================

    cnn.eval()

    cnn_to_mobilevit.hidden_cnn.eval()
    cnn_to_mobilevit.mobilevit_adapter.eval()
    final_channel_adapter.eval()
    hf_mobilevit.classifier.eval()

    val_loss = 0.0
    val_samples = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.float().to(device)

            # ------------------------------------------------
            # CNN
            # ------------------------------------------------

            cnn_features = cnn(images)

            # ------------------------------------------------
            # Hidden CNN
            # ------------------------------------------------

            hidden_features = (
                cnn_to_mobilevit.hidden_cnn(
                    cnn_features
                )
            )

            # ------------------------------------------------
            # 128 → 64
            # ------------------------------------------------

            features_64 = (
                cnn_to_mobilevit.mobilevit_adapter(
                    hidden_features
                )
            )

            # ------------------------------------------------
            # 64 → 3
            # ------------------------------------------------

            features_3 = (
                final_channel_adapter(
                    features_64
                )
            )

            # ------------------------------------------------
            # Resize
            # ------------------------------------------------

            mobilevit_input = F.interpolate(
                features_3,
                size=(256, 256),
                mode="bilinear",
                align_corners=False
            )

            # ------------------------------------------------
            # MobileViT
            # ------------------------------------------------

            outputs = hf_mobilevit(
                pixel_values=mobilevit_input
            )

            logits = outputs.logits.squeeze(1)

            # ------------------------------------------------
            # Loss
            # ------------------------------------------------

            loss = criterion(
                logits,
                labels
            )

            batch_size = images.size(0)

            val_loss += loss.item() * batch_size
            val_samples += batch_size

    val_loss /= val_samples


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "cnn_to_mobilevit":
                    cnn_to_mobilevit.state_dict(),

                "final_channel_adapter":
                    final_channel_adapter.state_dict(),

                "hf_mobilevit":
                    hf_mobilevit.state_dict(),

                "epoch":
                    epoch + 1,

                "val_loss":
                    val_loss
            },
            best_model_path
        )

        marker = " ★ BEST"

    else:

        marker = ""


    # ========================================================
    # PRINT
    # ========================================================

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
        f"{marker}"
    )


print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best validation loss: "
    f"{best_val_loss:.6f}"
)

print()
print("Best model saved to:")
print(best_model_path)

STARTING CNN → HIDDEN CNN → ADAPTER → MOBILEVIT TRAINING
Epoch [01/15] | Train Loss: 0.6756 | Val Loss: 0.5838 ★ BEST
Epoch [02/15] | Train Loss: 0.6250 | Val Loss: 0.5559 ★ BEST
Epoch [03/15] | Train Loss: 0.6114 | Val Loss: 0.5353 ★ BEST
Epoch [04/15] | Train Loss: 0.5944 | Val Loss: 0.6060
Epoch [05/15] | Train Loss: 0.5891 | Val Loss: 0.5481
Epoch [06/15] | Train Loss: 0.5875 | Val Loss: 0.5606
Epoch [07/15] | Train Loss: 0.5835 | Val Loss: 0.5377
Epoch [08/15] | Train Loss: 0.5692 | Val Loss: 0.5693
Epoch [09/15] | Train Loss: 0.5620 | Val Loss: 0.5677
Epoch [10/15] | Train Loss: 0.5542 | Val Loss: 0.5413
Epoch [11/15] | Train Loss: 0.5303 | Val Loss: 0.5347 ★ BEST
Epoch [12/15] | Train Loss: 0.5208 | Val Loss: 0.5663
Epoch [13/15] | Train Loss: 0.4956 | Val Loss: 0.5296 ★ BEST
Epoch [14/15] | Train Loss: 0.4822 | Val Loss: 0.5544
Epoch [15/15] | Train Loss: 0.4474 | Val Loss: 0.6303

TRAINING COMPLETE
Best validation loss: 0.529648

Best model saved to:
/content/drive/MyDrive/bes

In [ ]:
# ============================================================
# FINAL TEST / EVALUATION
# CNN → HIDDEN CNN → 128→64 → 64→3 → MOBILEVIT
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# STEP 1 — Load BEST combined checkpoint
# ------------------------------------------------------------

BEST_MODEL_PATH = "/content/drive/MyDrive/best_cnn_hidden_mobilevit.pth"

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

print("=" * 70)
print("LOADING BEST CNN → HIDDEN CNN → MOBILEVIT MODEL")
print("=" * 70)

# Load Hidden CNN
cnn_to_mobilevit.load_state_dict(
    checkpoint["cnn_to_mobilevit"]
)

# Load 64 → 3 adapter
final_channel_adapter.load_state_dict(
    checkpoint["final_channel_adapter"]
)

# Load Hugging Face MobileViT
hf_mobilevit.load_state_dict(
    checkpoint["hf_mobilevit"]
)

print("Checkpoint loaded successfully.")

if "epoch" in checkpoint:
    print("Best epoch:", checkpoint["epoch"])

if "val_loss" in checkpoint:
    print("Best validation loss:", checkpoint["val_loss"])


# ------------------------------------------------------------
# STEP 2 — Evaluation mode
# ------------------------------------------------------------

cnn.eval()
cnn_to_mobilevit.eval()
final_channel_adapter.eval()
hf_mobilevit.eval()


# ------------------------------------------------------------
# STEP 3 — Run TEST SET
# ------------------------------------------------------------

all_labels = []
all_probs = []
all_preds = []

with torch.no_grad():

    for images, labels in test_loader:

        # ----------------------------------------------------
        # Original input
        # Shape: [B, 11, 250, 250]
        # ----------------------------------------------------

        images = images.to(device)
        labels = labels.to(device)

        # ----------------------------------------------------
        # CNN
        # 11 → 64
        # Expected:
        # [B, 64, 31, 31]
        # ----------------------------------------------------

        cnn_features = cnn(images)

        # ----------------------------------------------------
        # Hidden CNN
        # 64 → 128
        # ----------------------------------------------------

        hidden_features = cnn_to_mobilevit.hidden_cnn(
            cnn_features
        )

        # ----------------------------------------------------
        # Adapter
        # 128 → 64
        # ----------------------------------------------------

        features_64 = cnn_to_mobilevit.mobilevit_adapter(
            hidden_features
        )

        # ----------------------------------------------------
        # Final adapter
        # 64 → 3
        # ----------------------------------------------------

        features_3 = final_channel_adapter(
            features_64
        )

        # ----------------------------------------------------
        # Resize
        # 31 × 31 → 256 × 256
        # ----------------------------------------------------

        mobilevit_input = F.interpolate(
            features_3,
            size=(256, 256),
            mode="bilinear",
            align_corners=False
        )

        # ----------------------------------------------------
        # MobileViT
        # Input:
        # [B, 3, 256, 256]
        # ----------------------------------------------------

        outputs = hf_mobilevit(
            pixel_values=mobilevit_input
        )

        # ----------------------------------------------------
        # Binary classification
        # ----------------------------------------------------

        logits = outputs.logits

        # Handle [B, 1] output
        if logits.ndim == 2 and logits.shape[1] == 1:

            logits = logits.squeeze(1)

            probs = torch.sigmoid(logits)

        # Handle [B, 2] output
        else:

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1]

        preds = (probs >= 0.5).long()

        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )


# ------------------------------------------------------------
# STEP 4 — Convert to NumPy
# ------------------------------------------------------------

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_preds = np.array(all_preds)


# ------------------------------------------------------------
# STEP 5 — Calculate metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_preds
)

precision = precision_score(
    all_labels,
    all_preds,
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_preds,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_preds,
    zero_division=0
)

try:

    roc_auc = roc_auc_score(
        all_labels,
        all_probs
    )

except ValueError:

    roc_auc = float("nan")


cm = confusion_matrix(
    all_labels,
    all_preds
)


# ------------------------------------------------------------
# STEP 6 — Print final results
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Number of test samples : {len(all_labels)}")
print()

print(f"Accuracy               : {accuracy:.4f}")
print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1 Score               : {f1:.4f}")
print(f"ROC-AUC                : {roc_auc:.4f}")

print()
print("Confusion Matrix:")
print(cm)

print()
print("=" * 70)
print("TESTING COMPLETE")
print("=" * 70)

LOADING BEST CNN → HIDDEN CNN → MOBILEVIT MODEL
Checkpoint loaded successfully.
Best epoch: 13
Best validation loss: 0.5296484979571165

FINAL TEST RESULTS
Number of test samples : 545

Accuracy               : 0.6734
Precision              : 0.4684
Recall                 : 0.2139
F1 Score               : 0.2937
ROC-AUC                : 0.6159

Confusion Matrix:
[[330  42]
 [136  37]]

TESTING COMPLETE
